<h1>Rappel sur les machines à états finis, MEF (Finite State Machine, FSM)</h1>

Une MEF nous permet de décrire un système plus ou moins complexe à l'aide de quelques **composantes simples**.

<h1>Composantes d'une MEF</h1>

![fsm_review](./images/fsm_review.png)

<span><b>États (States)</b></span>
<br/>
Les « modes » possibles du système. Le système ne peut être que dans un état à la fois.
<br/>

<span><b>Entrées (Inputs, Triggers)</b></span>
<br/>
Ce qui arrive au système (ex. pièce insérée, bouton pressé).
<br/>

<span><b>Transitions</b></span>
<br/>
Règles qui gouvernent le passage entre états (ex. si on reçoit l'entrée X en état A, aller à l’état B).
<br/>

<span><b>Sorties (Outputs)</b></span>
<br/>
Ce que le système fait (ex. commencer une poursuite, réapparaitre au centre de la carte).

![fsm_review](./images/fsm_review.png)


<span style="color:#11d500"><b>États (States)</b></span>
<br/>
Chasse, Mort, Fuite.
<br/>

<span style="color:#f6ed05"><b>Entrées (Inputs)</b></span>
<br/>
Début du jeu, Pacman powerup actif, Powerup terminé, Collision avec Pacman, Point de respawn.
<br/>

<span style="color:#009dd4"><b>Transitions</b></span>
<br/>
À l'état "Chasse", l'entrée "Pacman powerup actif" mène vers l'état "Fuite".
<br/>
À l'état "Fuite", l'entrée "Powerup terminé" mène vers l'état "Chasse".
<br/>
À l'état "Fuite", l'entrée "Collision avec Pacman" mène vers l'état "Mort".
<br/>
À l'état "Mort", l'entrée "Point de respawn" mène vers l'état "Chasse".
<br/>

<span style="color:#f75e5e"><b>Sorties (Outputs)</b></span>
<br/>
Le fantôme chasse Pacman, le fantôme fuit Pacman, le fantôme rentre au point de respawn.

<h1>Plusieurs manières d'implémenter une MEF avec Python</h1>

<h2>Sous forme de dictionnaire</h2>

In [ ]:
fsm = {
    "chasse": {
        "pacman powerup": "fuite"
    },
    "fuite": {
        "fin powerup": "chasse",
        "pacman attrape": "mort"
    },
    "mort": {
        "respawn atteint": "chasse"
    }
}

- Chaque état possible est une clé dans le dictionnaire parent.
- Chaque valeur est un sous-dictionnaire qui représente les transitions possibles.
- Par exemple, dans l'état "fuite", si l'entrée est "fin powerup", on passe à l'état "chasse". Mais si l'entrée est "pacman attrape", on passe à l'état "mort".

<h2>À l'aide d'une classe et d'énumérations</h2>

In [ ]:
from enum import Enum
import os, sys
sys.path.append(os.path.join(os.getcwd(), "..", ".."))
from ai.fsm.core import Machine

Les énumérations permettent de mieux gérer les états et les entrées que des chaînes de caractères.

In [ ]:
class Etats(Enum):
    EN_ATTENTE = 0
    CHASSE = 1
    FUITE = 2
    MORT = 3

class Entrees(Enum):
    JEU_COMMENCE = 0
    PACMAN_POWERUP = 1
    FIN_POWERUP = 2
    PACMAN_ATTRAPE = 3
    RESPAWN_ATTEINT = 4
    FIN_JEU = 5

Toute la logique derrière une MEF peut être cachée (encapsulée) derrière une classe.

In [ ]:
fantome_fsm = Machine(states=Etats, initial_state=Etats.EN_ATTENTE)

# On ajoute les transitions une par une
fantome_fsm.add_transition(
    sources=Etats.CHASSE,
    dest=Etats.FUITE,
    trigger=Entrees.PACMAN_POWERUP
)
fantome_fsm.add_transition(
    sources=Etats.FUITE,
    dest=Etats.CHASSE,
    trigger=Entrees.FIN_POWERUP
)
fantome_fsm.add_transition(
    sources=Etats.FUITE,
    dest=Etats.MORT,
    trigger=Entrees.PACMAN_ATTRAPE
)
fantome_fsm.add_transition(
    sources=Etats.MORT,
    dest=Etats.CHASSE,
    trigger=Entrees.RESPAWN_ATTEINT
)

In [ ]:
# On ajoute les transitions pour arrêter et recommencer le jeu
fantome_fsm.add_transition(
    sources=[Etats.CHASSE, Etats.FUITE, Etats.MORT],
    dest=Etats.EN_ATTENTE,
    trigger=Entrees.FIN_JEU
)
fantome_fsm.add_transition(
    sources=Etats.EN_ATTENTE,
    dest=Etats.CHASSE,
    trigger=Entrees.JEU_COMMENCE
)

La classe **Machine** nous permet aussi de visualiser notre MEF! L'**état actuel** est représenté par un **double cercle**.

In [ ]:
display(fantome_fsm.to_graphviz())

![fsm_code](./images/fsm_code.svg)

<h1>Retour sur la partie MEF du parc</h1>

Pour rendre intelligent le processus de prise de décision des robots dans la ronde 2.0, on vous a demandé d'implémenter une MEF à l'aide de la classe **Machine**, et en utilisant les états et les entrées fournis dans le fichier **"homework/fsm.py"**.

![fsm_park](./images/fsm_park.png)

In [ ]:
from __future__ import annotations

from enum import Enum, IntFlag

from ai.fsm.core import Machine

In [ ]:
class RobotState(Enum):
    ROAMING = "roaming"
    PICK_VISITOR = "pick_visitor"
    PICK_RIDE = "pick_ride"
    CHARGING = "charging"


class RobotTrigger(IntFlag):
    VISITOR_IN_QUEUE = 1 << 0
    VISITOR_ON_BOARD = 1 << 1
    VISITOR_DROP_OFF = 1 << 2
    LOW_BATTERY = 1 << 3
    FULL_BATTERY = 1 << 4

On demandait de compléter la fonction **get_robot_fsm** dans **"homework/fsm.py"**. La **Machine** doit être créer correctement en spécifiant les **états possibles** et **l'état initial**. Ensuite, il faut ajouter les **transitions** à l'aide de la fonction **add_transition** et en suivant le schéma fourni.

In [ ]:
fsm = Machine(states=RobotState, initial_state=RobotState.ROAMING)

In [ ]:
fsm.add_transition(
    sources=RobotState.ROAMING,
    dest=RobotState.PICK_VISITOR,
    trigger=RobotTrigger.VISITOR_IN_QUEUE
)
fsm.add_transition(
    sources=RobotState.PICK_VISITOR,
    dest=RobotState.PICK_RIDE,
    trigger=RobotTrigger.VISITOR_ON_BOARD
)
fsm.add_transition(
    sources=RobotState.PICK_RIDE,
    dest=RobotState.ROAMING,
    trigger=RobotTrigger.VISITOR_DROP_OFF
)

In [ ]:
# On peut ajouter la même transition pour plusieurs états sources
fsm.add_transition(
    sources=[
        RobotState.ROAMING,
        RobotState.PICK_VISITOR,
        RobotState.PICK_RIDE
    ],
    dest=RobotState.CHARGING,
    trigger=RobotTrigger.LOW_BATTERY
)

In [ ]:
fsm.add_transition(
    sources=RobotState.CHARGING,
    dest=RobotState.ROAMING,
    trigger=RobotTrigger.FULL_BATTERY
)

# Pour mieux comprendre l'utilisation du OU binaire (opérateur |),
# voir les notes sur RobotTrigger en commentaire de la fonction
fsm.add_transition(
    sources=RobotState.CHARGING,
    dest=RobotState.PICK_RIDE,
    trigger=(
        RobotTrigger.FULL_BATTERY
        | RobotTrigger.VISITOR_ON_BOARD
    )
)

Lien vers la solution: https://tinyurl.com/4te5eyc4

![fsm_goal](./images/fsm_goal.svg)